# Goal

What is most effective HP.ppo.consistency_coef / HP.ppo.prediction_coef?

# Notes

...

# Results

Bad experiment: loose goal, drastic diffs from 10/3 (no annealing, different global_steps_count, reduced obs_sequence_length) - not clear what went wrong.

In [ ]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    HP.system.random_seed = 42
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    
    HP.env.ident = 'FrostbiteNoFrameskip-v4'
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6
    
    HP.agent.parent = None # e.g. 17e_ppo_tr_atari_mp_10:3
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.obs_sequence_length = 4 # length observation chain agent incepts
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.positional_encoding = 'learned' # positional encoding type of the transformer: "", "absolute", "learned"
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    
    # Training procedure params (PPO related) 
    HP.ppo.curriculum = 'mix1'
    
    HP.ppo.global_steps_count = 9_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.tau = 'const(0.5)' # temperature to inject randomness during actions selection (Gumbel Max)
    
    HP.ppo.epochs_count = 2 
    HP.ppo.minibatches_count = 8
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    HP.ppo.ent_coef = 'const(0.05)' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = optuna_trial.suggest_float('ppo.consistency_coef', 0.0, 0.5)
    HP.ppo.prediction_coef = optuna_trial.suggest_float('ppo.prediction_coef', 0.0, 0.5)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # e target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP